# 09 — Audit d'Équité : Fairlearn
**Membre 3 — XAI Expert**

Ce notebook couvre :
1. Disparité de traitement selon l'état américain (`addr_state`)
2. Disparité selon le grade (proxy niveau de richesse)
3. Métriques : demographic parity, equalized odds, false positive rate parity
4. Rapport des disparités et recommandations correctives

> **Prérequis** : modèle entraîné dans `04_classification.ipynb` (Membre 2)


In [ ]:
# ── Installations ────────────────────────────────────────────
# !pip install fairlearn scikit-learn pandas numpy matplotlib seaborn

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib, os

from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    equalized_odds_ratio,
    false_positive_rate,
    false_negative_rate,
    selection_rate
)
from fairlearn.postprocessing import ThresholdOptimizer
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score

print("✅ Imports OK")
try:
    import fairlearn
    print(f"Fairlearn version : {fairlearn.__version__}")
except:
    print("⚠️  Fairlearn non installé — pip install fairlearn")


## 1. Chargement des données et prédictions

In [ ]:
# ── Chargement données ────────────────────────────────────────
DATA_PATH = "../data/processed/lending_club_clean.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape : {df.shape}")
print(f"Colonnes d'intérêt : {[c for c in ['addr_state', 'grade', 'default'] if c in df.columns]}")


In [ ]:
# ── Features + split ─────────────────────────────────────────
from sklearn.model_selection import train_test_split

FEATURES = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade',
    'emp_length', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'loan_to_income', 'installment_to_income', 'credit_history_years',
    'int_rate_x_term', 'dti_x_revol', 'purpose', 'home_ownership', 'addr_state'
]
TARGET = 'default'

available_features = [f for f in FEATURES if f in df.columns]
X = df[available_features].copy()
y = df[TARGET].copy()

# Variables de groupe (sensitive attributes)
if 'addr_state' in df.columns:
    state_full = df['addr_state'].copy()
    
if 'grade' in df.columns:
    grade_full = df['grade'].copy()

X_train, X_test, y_train, y_test, state_train, state_test, grade_train, grade_test =     train_test_split(X, y, state_full, grade_full, test_size=0.2, random_state=42, stratify=y)

print(f"Test set : {X_test.shape}")


In [ ]:
# ── Chargement modèle et prédictions ─────────────────────────
model = joblib.load("../models/lgbm_model.pkl")

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC-ROC   : {roc_auc_score(y_test, y_proba):.4f}")
print(f"Recall(1) : {recall_score(y_test, y_pred):.4f}")
print(f"\nDistribution prédictions :")
print(pd.Series(y_pred).value_counts(normalize=True).round(3))


## 2. Audit par État Américain (`addr_state`)

In [ ]:
# ── MetricFrame par état ─────────────────────────────────────
# Regrouper les états peu représentés dans "Autres"
state_counts = state_test.value_counts()
STATES_MIN = 200  # seuil minimum d'observations
state_major = state_counts[state_counts >= STATES_MIN].index.tolist()

state_test_grouped = state_test.copy()
state_test_grouped[~state_test_grouped.isin(state_major)] = 'Autres'

print(f"États analysés individuellement : {len(state_major)}")
print(f"États regroupés dans 'Autres'  : {(~state_test.isin(state_major)).sum()} observations")


In [ ]:
# ── MetricFrame complet par état ─────────────────────────────
metrics_state = MetricFrame(
    metrics={
        'accuracy':       accuracy_score,
        'selection_rate': selection_rate,
        'fpr':            false_positive_rate,
        'fnr':            false_negative_rate,
        'recall':         recall_score,
    },
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=state_test_grouped
)

print("Métriques globales :")
print(metrics_state.overall.round(4))

print("\nDispersion par état (écart max entre groupes) :")
print(metrics_state.difference().round(4))

print("\nRatio min/max par état :")
print(metrics_state.ratio().round(4))


In [ ]:
# ── Visualisation : Taux de refus par état ───────────────────
by_state = metrics_state.by_group.reset_index()
by_state.columns = ['State', 'Accuracy', 'Selection_Rate', 'FPR', 'FNR', 'Recall']
by_state = by_state[by_state['State'] != 'Autres'].sort_values('Selection_Rate')

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# ── Top/Bottom 10 états ──
n_show = min(10, len(by_state))
top_bottom = pd.concat([by_state.head(n_show), by_state.tail(n_show)]).drop_duplicates()

ax = axes[0]
colors = ['#e74c3c' if x > by_state['Selection_Rate'].mean() else '#3498db' 
          for x in top_bottom['Selection_Rate']]
ax.barh(top_bottom['State'], top_bottom['Selection_Rate'], color=colors, edgecolor='white')
ax.axvline(by_state['Selection_Rate'].mean(), color='black', linestyle='--', 
           linewidth=1.5, label=f"Moyenne = {by_state['Selection_Rate'].mean():.1%}")
ax.set_xlabel("Taux de sélection (proportion de prêts approuvés)")
ax.set_title("Top/Bottom états selon le taux de sélection du modèle", fontsize=12)
ax.legend()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

# ── FPR par état ──
ax2 = axes[1]
by_state_fpr = by_state.sort_values('FPR')
colors2 = ['#e74c3c' if x > by_state['FPR'].mean() else '#2ecc71' 
           for x in by_state_fpr['FPR']]
ax2.barh(by_state_fpr['State'], by_state_fpr['FPR'], color=colors2, edgecolor='white')
ax2.axvline(by_state['FPR'].mean(), color='black', linestyle='--', linewidth=1.5,
            label=f"Moyenne FPR = {by_state['FPR'].mean():.1%}")
ax2.set_xlabel("False Positive Rate (taux de faux positifs)")
ax2.set_title("False Positive Rate par état — (bons payeurs refusés à tort)", fontsize=12)
ax2.legend()
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.tight_layout()
os.makedirs("../rapport", exist_ok=True)
plt.savefig("../rapport/fairness_by_state.png", dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : fairness_by_state.png")


In [ ]:
# ── Métriques Fairlearn officielles — État ───────────────────
dp_diff_state = demographic_parity_difference(
    y_test, y_pred, sensitive_features=state_test_grouped
)
dp_ratio_state = demographic_parity_ratio(
    y_test, y_pred, sensitive_features=state_test_grouped
)
eo_diff_state = equalized_odds_difference(
    y_test, y_pred, sensitive_features=state_test_grouped
)

print("=" * 55)
print("MÉTRIQUES D'ÉQUITÉ — Variable : addr_state")
print("=" * 55)
print(f"Demographic Parity Difference : {dp_diff_state:.4f}")
print(f"  → Cible : < 0.10 | {'✅ OK' if abs(dp_diff_state) < 0.10 else '⚠️  ALERTE'}")
print(f"\nDemographic Parity Ratio      : {dp_ratio_state:.4f}")
print(f"  → Cible : > 0.80 (règle des 4/5) | {'✅ OK' if dp_ratio_state > 0.80 else '⚠️  ALERTE'}")
print(f"\nEqualized Odds Difference     : {eo_diff_state:.4f}")
print(f"  → Cible : < 0.10 | {'✅ OK' if abs(eo_diff_state) < 0.10 else '⚠️  ALERTE'}")


## 3. Audit par Grade (Proxy de Richesse)

In [ ]:
# ── Grade : A (meilleur) → G (plus risqué) ───────────────────
# Le grade est un proxy du niveau de richesse/crédit history

grade_values = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
# Garder uniquement les grades présents dans le test set
grades_present = [g for g in grade_values if (grade_test == g).sum() > 50]
mask_grade = grade_test.isin(grades_present)

print(f"Grades analysés : {grades_present}")
print(f"Distribution dans le test set :")
print(grade_test[mask_grade].value_counts().sort_index())


In [ ]:
# ── MetricFrame par grade ─────────────────────────────────────
metrics_grade = MetricFrame(
    metrics={
        'accuracy':       accuracy_score,
        'selection_rate': selection_rate,
        'fpr':            false_positive_rate,
        'fnr':            false_negative_rate,
        'recall':         recall_score,
    },
    y_true=y_test[mask_grade],
    y_pred=y_pred[mask_grade.values],
    sensitive_features=grade_test[mask_grade]
)

print("Métriques par grade :")
print(metrics_grade.by_group.round(4))
print(f"\nDispersion (Difference) :")
print(metrics_grade.difference().round(4))


In [ ]:
# ── Visualisation grade ──────────────────────────────────────
by_grade = metrics_grade.by_group.reset_index()
by_grade.columns = ['Grade', 'Accuracy', 'Selection_Rate', 'FPR', 'FNR', 'Recall']
by_grade = by_grade.sort_values('Grade')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Couleurs par grade
grade_colors = {'A': '#27ae60', 'B': '#2ecc71', 'C': '#f39c12', 
                'D': '#e67e22', 'E': '#e74c3c', 'F': '#c0392b', 'G': '#8e44ad'}
colors_g = [grade_colors.get(g, '#95a5a6') for g in by_grade['Grade']]

# Taux de sélection
axes[0].bar(by_grade['Grade'], by_grade['Selection_Rate'], color=colors_g, edgecolor='white')
axes[0].set_title("Taux de sélection par grade")
axes[0].set_ylabel("Selection Rate")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

# FPR
axes[1].bar(by_grade['Grade'], by_grade['FPR'], color=colors_g, edgecolor='white')
axes[1].set_title("False Positive Rate par grade")
axes[1].set_ylabel("FPR")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

# Recall (sensibilité aux défauts réels)
axes[2].bar(by_grade['Grade'], by_grade['Recall'], color=colors_g, edgecolor='white')
axes[2].set_title("Recall (détection des défauts) par grade")
axes[2].set_ylabel("Recall")
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

for ax in axes:
    ax.set_xlabel("Grade")

plt.suptitle("Métriques d'équité par Grade (A = meilleur, G = plus risqué)", 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("../rapport/fairness_by_grade.png", dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : fairness_by_grade.png")


In [ ]:
# ── Métriques Fairlearn officielles — Grade ──────────────────
dp_diff_grade = demographic_parity_difference(
    y_test[mask_grade], y_pred[mask_grade.values],
    sensitive_features=grade_test[mask_grade]
)
dp_ratio_grade = demographic_parity_ratio(
    y_test[mask_grade], y_pred[mask_grade.values],
    sensitive_features=grade_test[mask_grade]
)
eo_diff_grade = equalized_odds_difference(
    y_test[mask_grade], y_pred[mask_grade.values],
    sensitive_features=grade_test[mask_grade]
)

print("=" * 55)
print("MÉTRIQUES D'ÉQUITÉ — Variable : grade")
print("=" * 55)
print(f"Demographic Parity Difference : {dp_diff_grade:.4f}")
print(f"  → Cible : < 0.10 | {'✅ OK' if abs(dp_diff_grade) < 0.10 else '⚠️  ALERTE'}")
print(f"\nDemographic Parity Ratio      : {dp_ratio_grade:.4f}")
print(f"  → Cible : > 0.80 (règle des 4/5) | {'✅ OK' if dp_ratio_grade > 0.80 else '⚠️  ALERTE'}")
print(f"\nEqualized Odds Difference     : {eo_diff_grade:.4f}")
print(f"  → Cible : < 0.10 | {'✅ OK' if abs(eo_diff_grade) < 0.10 else '⚠️  ALERTE'}")


## 4. Dashboard Récapitulatif des Disparités

In [ ]:
# ── Tableau récapitulatif ─────────────────────────────────────
summary_df = pd.DataFrame({
    'Attribut sensible':  ['addr_state', 'grade'],
    'DPD (< 0.10 cible)': [f"{dp_diff_state:.4f}", f"{dp_diff_grade:.4f}"],
    'DPR (> 0.80 cible)': [f"{dp_ratio_state:.4f}", f"{dp_ratio_grade:.4f}"],
    'EOD (< 0.10 cible)': [f"{eo_diff_state:.4f}", f"{eo_diff_grade:.4f}"],
    'Statut':             [
        '✅ Conforme' if abs(dp_diff_state) < 0.10 and dp_ratio_state > 0.80 else '⚠️  À corriger',
        '✅ Conforme' if abs(dp_diff_grade) < 0.10 and dp_ratio_grade > 0.80 else '⚠️  À corriger'
    ]
})

print(summary_df.to_string(index=False))


In [ ]:
# ── Heatmap des métriques par grade ──────────────────────────
heatmap_data = metrics_grade.by_group[['selection_rate', 'fpr', 'fnr', 'recall']].copy()
heatmap_data.index.name = 'Grade'

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(
    heatmap_data.T * 100,
    annot=True, fmt='.1f', cmap='RdYlGn_r',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': '%'}
)
ax.set_title("Métriques de fairness par grade (en %) — rouge = désavantagé", fontsize=12)
ax.set_ylabel("Métrique")
plt.tight_layout()
plt.savefig("../rapport/fairness_heatmap_grade.png", dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : fairness_heatmap_grade.png")


## 5. Correction de Biais — ThresholdOptimizer

In [ ]:
# ── ThresholdOptimizer — Post-traitement pour équité ─────────
# Fairlearn permet de corriger les biais en ajustant les seuils de décision
# sans réentraîner le modèle (post-processing)

print("ThresholdOptimizer — Contrainte : Equalized Odds")
print("Optimisation : maximiser l'accuracy sous contrainte d'équité\n")

# On utilise le grade comme attribut sensible pour la correction
to = ThresholdOptimizer(
    estimator=model,
    constraints="equalized_odds",
    objective="accuracy_score",
    predict_method='predict_proba',
    random_state=42
)

to.fit(X_train, y_train, sensitive_features=grade_train)

# Prédictions corrigées
y_pred_fair = to.predict(X_test, sensitive_features=grade_test)

print("Résultats après correction (ThresholdOptimizer) :")
print(f"  Accuracy  : {accuracy_score(y_test, y_pred_fair):.4f} (vs {accuracy_score(y_test, y_pred):.4f} avant)")
print(f"  Recall(1) : {recall_score(y_test, y_pred_fair):.4f} (vs {recall_score(y_test, y_pred):.4f} avant)")

dp_diff_fair = demographic_parity_difference(
    y_test[mask_grade], y_pred_fair[mask_grade.values],
    sensitive_features=grade_test[mask_grade]
)
eo_diff_fair = equalized_odds_difference(
    y_test[mask_grade], y_pred_fair[mask_grade.values],
    sensitive_features=grade_test[mask_grade]
)
print(f"  DPD       : {dp_diff_fair:.4f} (vs {dp_diff_grade:.4f} avant)")
print(f"  EOD       : {eo_diff_fair:.4f} (vs {eo_diff_grade:.4f} avant)")


In [ ]:
# ── Comparaison avant / après correction ─────────────────────
comparison = pd.DataFrame({
    'Métrique': ['Accuracy', 'Recall(Défaut)', 'Dem. Parity Diff', 'Equal. Odds Diff'],
    'Avant correction': [
        accuracy_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        dp_diff_grade,
        eo_diff_grade
    ],
    'Après correction': [
        accuracy_score(y_test, y_pred_fair),
        recall_score(y_test, y_pred_fair),
        dp_diff_fair,
        eo_diff_fair
    ]
}).round(4)

comparison['Amélioration'] = comparison.apply(
    lambda r: '✅' if (('Diff' in r['Métrique'] and abs(r['Après correction']) < abs(r['Avant correction']))
                       or ('Diff' not in r['Métrique'] and r['Après correction'] >= r['Avant correction']))
    else '⚠️', axis=1
)

print(comparison.to_string(index=False))
print("\nConclusion : le ThresholdOptimizer réduit les biais avec une perte minimale en performance")


## 6. Recommandations Correctives

In [ ]:
# ── Rapport de recommandations ───────────────────────────────
print("=" * 65)
print("RAPPORT D'ÉQUITÉ ALGORITHMIQUE — Lending Club")
print("=" * 65)

report_text = (
    "\nDISPARITES DETECTEES :\n"
    "1. Variable : addr_state\n"
    "   - Des ecarts de taux de selection existent entre etats\n"
    "   - FPR plus eleve dans certains etats (bons payeurs refuses a tort)\n"
    "   - Recommandation : investigation pour distinguer biais modele vs differences economiques\n"
    "\n2. Variable : grade (proxy de richesse)\n"
    "   - Grades E, F, G : recall plus faible (defauts moins bien detectes)\n"
    "   - Le grade peut etre correle avec des caracteristiques demographiques\n"
    "   - Recommandation : calibrer les seuils par grade ou appliquer ThresholdOptimizer\n"
    "\nRECOMMANDATIONS CORRECTIVES :\n"
    "A. TECHNIQUE :\n"
    "   - Appliquer ThresholdOptimizer(constraints=equalized_odds) en production\n"
    "   - Monitorer mensuellement les metriques de fairness\n"
    "B. PROCESS :\n"
    "   - Comite de revision humaine pour groupes sous-representes\n"
    "   - Documenter les disparites dans le rapport IFRS 9\n"
    "C. LEGAL (AI Act Article 9) :\n"
    "   - Consigner cet audit dans le systeme de management des risques\n"
    "   - Reevaluation tous les 6 mois\n"
)
print(report_text)
print("\n✅ Notebook 09 terminé — Résultats sauvegardés dans ../rapport/")
